In [33]:
import torch
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, CrossEntropyLoss
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam
from sklearn.model_selection import StratifiedKFold

In [34]:
class MyDataset(Dataset):
    def __init__(self, datas: np.ndarray, labels: np.ndarray) -> None:
        self._x = torch.from_numpy(datas).float()
        self._y = torch.from_numpy(labels).long()

    def __len__(self) -> int:
        return len(self._x)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        return self._x[idx], self._y[idx]

In [35]:
def train_model_class(features, targets, kf, batch, epochs):
    accuracy_scores = []
    f1_scores = []

    for i, (train_index, test_index) in enumerate(kf.split(features, targets)):
        print(f"\nTreinando modelo ----- Fold {i + 1}")
        # Criar um modelo novo para cada fold
        model = Sequential(
            Linear(4, 32),
            ReLU(),
            Linear(32, 16),
            ReLU(),
            Linear(16, 3)
        )
        optmizer = Adam(model.parameters(), lr=0.001)
        criterion = CrossEntropyLoss()
        # Preparar os dados de treinamento

        train_data = MyDataset(features[train_index], targets[train_index])
        train_loader = DataLoader(train_data, batch_size=batch)

        # Treinar o modelo
        for i_epochs in range(epochs):
            model.train()

            for x, y in train_loader:
                optmizer.zero_grad()

                output = model(x)
                loss = criterion(output, y)

                loss.backward()
                optmizer.step()

        # Avaliar o modelo
        model.eval()
        with torch.no_grad():
            x_test_tensor = Tensor(features[test_index]).float()
            y_true = targets[test_index]
            result = model(x_test_tensor)
            y_pred = result.argmax(dim=1).cpu().numpy()
            accuracy_s = accuracy_score(y_true, y_pred)
            f1_s = f1_score(y_true, y_pred, average="macro")

            accuracy_scores.append(accuracy_s)
            f1_scores.append(f1_s)

            print(f"Accuracy: {accuracy_s:.4f}")
            print(f"F1-score: {f1_s:.4f}")
            print("Matriz de confusão:")
            print(confusion_matrix(y_true, y_pred))

    print(
        f"\nResultado médio do modelo ----- "
        f"Accuracy: {np.mean(accuracy_scores):.4f}, "
        f"F1-score: {np.mean(f1_scores):.4f}"
    )

In [36]:
dataset = load_iris()

features = dataset.data
targets = dataset.target

In [37]:
# Configurações do treinamento
max_epochs = 500
batch = 8
# Configurar a validação cruzada
kf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Executar o treinamento e a avaliação
train_model_class(features, targets,kf,batch, max_epochs)



Treinando modelo ----- Fold 1
Accuracy: 1.0000
F1-score: 1.0000
Matriz de confusão:
[[10  0  0]
 [ 0 10  0]
 [ 0  0 10]]

Treinando modelo ----- Fold 2
Accuracy: 1.0000
F1-score: 1.0000
Matriz de confusão:
[[10  0  0]
 [ 0 10  0]
 [ 0  0 10]]

Treinando modelo ----- Fold 3
Accuracy: 0.9333
F1-score: 0.9327
Matriz de confusão:
[[10  0  0]
 [ 0  8  2]
 [ 0  0 10]]

Treinando modelo ----- Fold 4
Accuracy: 0.9667
F1-score: 0.9666
Matriz de confusão:
[[10  0  0]
 [ 0 10  0]
 [ 0  1  9]]

Treinando modelo ----- Fold 5
Accuracy: 0.9667
F1-score: 0.9666
Matriz de confusão:
[[10  0  0]
 [ 0 10  0]
 [ 0  1  9]]

Resultado médio do modelo ----- Accuracy: 0.9733, F1-score: 0.9732
